# 🍼 Notebook Prapemrosesan & Pembersihan Data (Pabrik Susu Bayi)

Notebook ini dirancang untuk melakukan pembersihan data (*data cleaning*) dan rekayasa fitur (*feature engineering*) secara terintegrasi pada **10 dataset** yang berasal dari **Pabrik Susu Bayi dengan Robot Gudang**.

Hasil dari prapemrosesan ini akan diekspor menjadi 3 dataset bersih terpisah untuk melatih model Machine Learning pada 3 area analisis utama:
1. **🤖 Predictive Maintenance**: Memprediksi potensi *error* robot otonom berdasarkan riwayat kerja dan daya baterai.
2. **🧪 Optimasi Kontrol Kualitas Susu**: Menganalisis ambang batas aman pengeringan/nutrisi untuk mencegah kontaminasi bakteri/kimia.
3. **📦 Logistics ETA & Cost Optimization**: Memprediksi rute pengiriman rentan *delay* dan meminimalkan biaya rantai pasok regional.

## 🛠️ Langkah 1: Inisialisasi & Impor Pustaka

Kami akan mengimpor pustaka analisis data standar industri (`pandas`, `numpy`) serta menetapkan direktori ruang kerja.

In [1]:
import os
import pandas as pd
import numpy as np

# Gunakan path relatif dari ruang kerja saat ini
BASE_DIR = "."
OUTPUT_DIR = os.path.join(BASE_DIR, "processed_data")

# Buat direktori output untuk data bersih jika belum ada
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Direktori output bersih siap di: {OUTPUT_DIR}")

Direktori output bersih siap di: .\processed_data


## 📂 Langkah 2: Memuat 10 Dataset Manufaktur & Logistik

Kami memuat seluruh dataset master dan transaksi untuk diinspeksi.

In [2]:
print("Memuat dataset dari direktori master...")
df_robots = pd.read_csv(os.path.join(BASE_DIR, "ds1_robots.csv"))
df_robot_tasks = pd.read_csv(os.path.join(BASE_DIR, "ds1_robot_tasks.csv"))
df_products = pd.read_csv(os.path.join(BASE_DIR, "ds2_products.csv"))
df_production_orders = pd.read_csv(os.path.join(BASE_DIR, "ds2_production_orders.csv"))
df_storage_locations = pd.read_csv(os.path.join(BASE_DIR, "ds3_storage_locations.csv"))
df_inventory_movements = pd.read_csv(os.path.join(BASE_DIR, "ds3_inventory_movements.csv"))
df_batches = pd.read_csv(os.path.join(BASE_DIR, "ds4_batches.csv"))
df_quality_inspections = pd.read_csv(os.path.join(BASE_DIR, "ds4_quality_inspections.csv"))
df_delivery_routes = pd.read_csv(os.path.join(BASE_DIR, "ds5_delivery_routes.csv"))
df_shipments = pd.read_csv(os.path.join(BASE_DIR, "ds5_shipments.csv"))

print("Semua dataset berhasil dimuat!")

Memuat dataset dari direktori master...
Semua dataset berhasil dimuat!


## 🤖 Bagian 1: Preprocessing untuk Predictive Maintenance Robot Gudang

Di sini, kami menggabungkan data master robot (`ds1_robots.csv`) dengan data log penugasan (`ds1_robot_tasks.csv`). Kami merekayasa fitur-fitur penting per robot:
- Jumlah total tugas yang didelegasikan (`total_tasks_attempted`).
- Jumlah tugas sukses, gagal, batal, dan parsial.
- Persentase kesuksesan tugas robot (`task_success_rate_pct`).
- Rata-rata durasi tugas dalam menit.
- Total daya baterai terkonsumsi dalam kWh.
- Jumlah muatan berat kumulatif yang ditangani.
- Total kemunculan error selama bertugas.
- Hari berlalu sejak pemeliharaan terakhir (`days_since_last_maintenance`).

In [3]:
print("Memulai preprocessing Predictive Maintenance...")

# 1. Konversi tanggal ke format datetime
date_cols_robots = ['purchase_date', 'installation_date', 'last_maintenance_date', 'next_maintenance_date']
for col in date_cols_robots:
    if col in df_robots.columns:
        df_robots[col] = pd.to_datetime(df_robots[col], errors='coerce')

date_cols_tasks = ['start_datetime', 'end_datetime']
for col in date_cols_tasks:
    if col in df_robot_tasks.columns:
        df_robot_tasks[col] = pd.to_datetime(df_robot_tasks[col], errors='coerce')

# 2. Agregasi transaksional tugas per robot
task_agg = df_robot_tasks.groupby('robot_id').agg(
    total_tasks_attempted=('task_id', 'count'),
    tasks_completed=('task_status', lambda x: (x == 'Completed').sum()),
    tasks_failed=('task_status', lambda x: (x == 'Failed').sum()),
    tasks_cancelled=('task_status', lambda x: (x == 'Cancelled').sum()),
    tasks_partial=('task_status', lambda x: (x == 'Partial').sum()),
    average_task_duration_min=('duration_minutes', 'mean'),
    total_energy_consumed_kwh=('energy_consumed_kwh', 'sum'),
    total_weight_handled_kg=('weight_kg', 'sum'),
    task_error_count=('error_occurred', 'sum')
).reset_index()

# 3. Tambahkan fitur tingkat kesuksesan tugas (%)
task_agg['task_success_rate_pct'] = (task_agg['tasks_completed'] / task_agg['total_tasks_attempted']) * 100

# 4. Merger dengan profil robot master
df_predictive_maintenance = pd.merge(df_robots, task_agg, on='robot_id', how='left')

# 5. Imputasi nilai kosong untuk robot yang tidak memiliki log tugas
fill_cols = [
    'total_tasks_attempted', 'tasks_completed', 'tasks_failed', 
    'tasks_cancelled', 'tasks_partial', 'average_task_duration_min', 
    'total_energy_consumed_kwh', 'total_weight_handled_kg', 
    'task_error_count', 'task_success_rate_pct'
]
df_predictive_maintenance[fill_cols] = df_predictive_maintenance[fill_cols].fillna(0)

# 6. Hitung selisih hari sejak pemeliharaan terakhir
df_predictive_maintenance['days_since_last_maintenance'] = (pd.Timestamp.now() - df_predictive_maintenance['last_maintenance_date']).dt.days

print(f"Preprocessing Predictive Maintenance Selesai! Shape: {df_predictive_maintenance.shape}")

Memulai preprocessing Predictive Maintenance...
Preprocessing Predictive Maintenance Selesai! Shape: (10000, 31)


## 🧪 Bagian 2: Preprocessing untuk Optimasi Kontrol Kualitas Susu

Pada bagian ini, kami menggabungkan data batch manufaktur (`ds4_batches.csv`) dengan data pengujian mutu laboratorium (`ds4_quality_inspections.csv`) dan katalog produk (`ds2_products.csv`). Fitur yang direkayasa:
- Selisih kadar protein hasil uji lab vs spesifikasi target produk (`protein_variance`).
- Selisih kadar lemak hasil uji lab vs spesifikasi target produk (`fat_variance`).
- Selisih kadar kelembaban bubuk hasil uji lab vs target batch (`moisture_variance`).
- Imputasi nilai kontaminasi kosong dengan label `'None'`.

In [4]:
print("Memulai preprocessing Kontrol Kualitas Susu...")

# 1. Konversi tanggal
for col in ['production_date', 'expiry_date', 'release_date']:
    if col in df_batches.columns:
        df_batches[col] = pd.to_datetime(df_batches[col], errors='coerce')

if 'inspection_datetime' in df_quality_inspections.columns:
    df_quality_inspections['inspection_datetime'] = pd.to_datetime(df_quality_inspections['inspection_datetime'], errors='coerce')

# 2. Gabungkan Inspeksi Lab dengan Batch Produksi
df_milk_quality = pd.merge(df_quality_inspections, df_batches, on='batch_id', suffixes=('_inspect', '_batch'), how='inner')

# 3. Gabungkan dengan Katalog Produk master
df_milk_quality = pd.merge(df_milk_quality, df_products, on='product_id', suffixes=('', '_product'), how='left')

# 4. Rekayasa Fitur Variansi Zat Gizi & Kelembaban
df_milk_quality['protein_variance'] = df_milk_quality['protein_result_pct'] - df_milk_quality['protein_content_pct']
df_milk_quality['fat_variance'] = df_milk_quality['fat_result_pct'] - df_milk_quality['fat_content_pct']
df_milk_quality['moisture_variance'] = df_milk_quality['moisture_result_pct'] - df_milk_quality['moisture_content_pct']

# 5. Imputasi tipe kontaminasi kosong
df_milk_quality['contamination_type'] = df_milk_quality['contamination_type'].fillna('None')

print(f"Preprocessing Kontrol Kualitas Susu Selesai! Shape: {df_milk_quality.shape}")

Memulai preprocessing Kontrol Kualitas Susu...
Preprocessing Kontrol Kualitas Susu Selesai! Shape: (10000, 61)


## 📦 Bagian 3: Preprocessing untuk Logistics ETA & Cost Optimization

Kami menggabungkan log pengapalan kontainer susu (`ds5_shipments.csv`) dengan data rute ekspedisi logistik (`ds5_delivery_routes.csv`). Fitur yang direkayasa:
- **Durasi Pengiriman Aktual** dalam hari (`actual_duration_days`).
- **Durasi Pengiriman Terjadwal** dalam hari (`scheduled_duration_days`).
- **Selisih/Variansi Keterlambatan** dalam hari (`delay_days`).
- Status keterlambatan biner (`is_delayed`) sebagai target pelatihan model.
- Ekstraksi fitur musiman (`shipment_month` dan `shipment_day_of_week`).
- Biaya pengiriman riil per kilogram (`real_cost_per_kg_usd`).
- Audit Cold Chain untuk mendeteksi kontainer pendingin yang malfungsi (`cold_chain_excursion_detected`).

In [5]:
print("Memulai preprocessing Logistics ETA...")

# 1. Konversi tanggal
date_cols_shipments = ['shipment_date', 'scheduled_delivery_date', 'actual_delivery_date', 'customs_clearance_date']
for col in date_cols_shipments:
    if col in df_shipments.columns:
        df_shipments[col] = pd.to_datetime(df_shipments[col], errors='coerce')

# 2. Gabungkan Pengiriman dengan Rute Logistik
df_logistics = pd.merge(df_shipments, df_delivery_routes, on='route_id', suffixes=('', '_route'), how='inner')

# 3. Rekayasa Fitur Pengiriman & Durasi
df_logistics['actual_duration_days'] = (df_logistics['actual_delivery_date'] - df_logistics['shipment_date']).dt.total_seconds() / (24 * 3600)
df_logistics['scheduled_duration_days'] = (df_logistics['scheduled_delivery_date'] - df_logistics['shipment_date']).dt.total_seconds() / (24 * 3600)
df_logistics['delay_days'] = df_logistics['actual_duration_days'] - df_logistics['scheduled_duration_days']

# Target biner untuk klasifikasi delay
df_logistics['is_delayed'] = (df_logistics['actual_delivery_date'] > df_logistics['scheduled_delivery_date']).astype(int)

# Fitur musiman
df_logistics['shipment_month'] = df_logistics['shipment_date'].dt.month
df_logistics['shipment_day_of_week'] = df_logistics['shipment_date'].dt.dayofweek

# Biaya riil per kg pengiriman cargo
df_logistics['real_cost_per_kg_usd'] = np.where(
    df_logistics['weight_kg'] > 0,
    df_logistics['total_cost_usd'] / df_logistics['weight_kg'],
    0
)

# Audit rantai dingin (Cold Chain Excursion)
df_logistics['cold_chain_excursion_detected'] = np.where(
    (df_logistics['cold_chain_required'] == 1) & (df_logistics['temperature_max_c'] > 8.0),
    1,
    0
)

# Tangani nilai kosong delay reason
df_logistics['delay_reason'] = df_logistics['delay_reason'].fillna('None')

print(f"Preprocessing Logistics ETA Selesai! Shape: {df_logistics.shape}")

Memulai preprocessing Logistics ETA...
Preprocessing Logistics ETA Selesai! Shape: (10000, 47)


## 💾 Bagian 4: Ekspor Dataset Bersih Siap Latih

Kami mengekspor ketiga data hasil preprocessing ke folder `processed_data/` dalam format CSV.

In [6]:
print("Mengekspor dataset bersih ter-preprocessing...")

pm_output_path = os.path.join(OUTPUT_DIR, "clean_predictive_maintenance.csv")
df_predictive_maintenance.to_csv(pm_output_path, index=False)

mq_output_path = os.path.join(OUTPUT_DIR, "clean_milk_quality.csv")
df_milk_quality.to_csv(mq_output_path, index=False)

log_output_path = os.path.join(OUTPUT_DIR, "clean_logistics_eta.csv")
df_logistics.to_csv(log_output_path, index=False)

print("--- Seluruh data bersih berhasil disimpan! ---")
print(f"1. Predictive Maintenance: {pm_output_path}")
print(f"2. Milk Quality Control:  {mq_output_path}")
print(f"3. Logistics ETA:         {log_output_path}")

Mengekspor dataset bersih ter-preprocessing...
--- Seluruh data bersih berhasil disimpan! ---
1. Predictive Maintenance: .\processed_data\clean_predictive_maintenance.csv
2. Milk Quality Control:  .\processed_data\clean_milk_quality.csv
3. Logistics ETA:         .\processed_data\clean_logistics_eta.csv
